Land-use change classification of a pixel by applying different rules 

Per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/6837cb74-6c60-800a-b108-fc89b2f766b8

In [1]:
import pandas as pd
import re

In [112]:
driver_code_map = {
    "permanent ag": "D1",
    "hard commodities": "D2",
    "shifting ag": "D3",
    "logging": "D4",
    "wildfire": "D5",
    "settlement": "D6",
    "other natural disturbances": "D7"
}

In [113]:
# Configuration
file_path = "/mnt/c/GIS/git/AFOLU_GHG_flux_model/src/LULUCF/scripts/postprocessing/LC_LU_scenarios.xlsx"  # Update if needed
sheet_name = "scenarios"

In [132]:
# Classification logic
def classify_by_regex(scenario_id, ts_list, regex_rules, driver=None):
   
    driver_str = str(driver).strip().lower() if pd.notna(driver) else ""
    driver_code = driver_code_map.get(driver_str, "")
        
    ts_str = '-'.join(map(str, ts_list))   # Creates the string of LC codes

    if driver_code:
        ts_str += f"-{driver_code}"
    
    print(f"Scenario {scenario_id}: {ts_str}")
    for i, (pattern, shifting_ag, LUC, conversion) in enumerate (regex_rules, start=1):
        if re.search(pattern, ts_str):
            return shifting_ag, LUC, conversion, i
    return False, False, False, None

In [111]:
# Iterate through scenarios and classify
def classify_dataframe_regex(df, regex_rules):
    results = []
    for idx, row in df.iterrows():
        driver = row.iloc[1]  # Driver
        ts_list = row.iloc[2:].tolist()  # Skip the first column (scenario ID)
        scenario_id = row.iloc[0]        # Use the actual scenario ID
        shifting_ag, luc_class, conversion_occurred, rule_number = classify_by_regex(scenario_id, ts_list, regex_rules, driver=driver)
        results.append({
            "shifting agriculture": shifting_ag,
            "LUC_class": luc_class,
            "conversion_occurred": conversion_occurred,
            "rule_number": rule_number
        })
    return pd.DataFrame(results)

In [165]:
# Load data
scenarios_df = pd.read_excel(file_path, sheet_name=sheet_name)

In [167]:
# Define regex-based LUC/conversion rules
regex_rules = [
    (r"^7(-7)*$", "not shifting ag",   "Forest remaining Forest",       "not conversion"),
    (r"^5(-5)*$", "not shifting ag",   "Grassland remaining Grassland",       "not conversion"),
    (r"^10(-10)*$", "not shifting ag",   "Cropland remaining Cropland",       "not conversion"),
    (r"^7(-7)*-D3$", "shifting ag",   "Cropland remaining Cropland",       "not conversion"),
    (r"(-\d+)*(-7){1,}(-\d+)*-10(-10){1,}(-\d+)*", "shifting ag",   "Forest to Cropland",       "not conversion"),
    (r"(-\d+)*(-5){1,}(-\d+)*-7(-7){1,}(-\d+)*",   "shifting ag",   "Forest to Cropland",       "not conversion"),
    ("^7(-7)*-\d+(-10)$", "not shifting ag", "Forest converted to cropland", "conversion"),
    ("^7(-7)*-\d+(-10)*-D3$", "shifting ag", "Cropland remaining cropland", "not conversion"),
    
]

In [166]:
# Run classification
results_df = classify_dataframe_regex(scenarios_df, regex_rules)

# Output results 
print("\nClassification Results:")
print(results_df)

Scenario 0: 7-7-7-7-7-7-7-7-7-7-7-7
Scenario 1: 5-5-5-5-5-5-5-5-5-5-5-5
Scenario 2: 10-10-10-10-10-10-10-10-10-10-10-10
Scenario 3: 7-7-7-7-7-7-7-7-7-7-10-10
Scenario 4: 7-7-7-7-7-7-7-7-7-7-10-10-D3
Scenario 5: 7-7-7-7-7-10-10-7-7-7-10-10-D3
Scenario 6: 10-10-7-7-7-7-10-7-7-7-10-10-D3
Scenario 7: 5-5-7-7-7-7-5-7-7-7-5-5-D3
Scenario 8: 7-10-7-10-7-7-10-10-10-10-10-10
Scenario 9: 7-7-7-7-7-7-7-7-7-7-7-7-D3

Classification Results:
  shifting agriculture                      LUC_class conversion_occurred  \
0      not shifting ag        Forest remaining Forest      not conversion   
1      not shifting ag  Grassland remaining Grassland      not conversion   
2      not shifting ag    Cropland remaining Cropland      not conversion   
3          shifting ag             Forest to Cropland      not conversion   
4          shifting ag             Forest to Cropland      not conversion   
5          shifting ag             Forest to Cropland      not conversion   
6          shifting ag      